# Student-load probe

`training_utils.py` never loaded the BaCP student from `--trained_weights`: the
call sat in the non-BaCP branch, so the arm kept the ImageNet weights
`_create_base_model` gives it -- and because `adapt_resnet_for_small_images`
replaces `conv1` with a fresh random stem *after* that load, BaCP started at
about 9% top-1 while I.P. started from the 91.8% dense model.

This probe runs the two flattest cells with the student loaded, changing
nothing else.

| cell | I.P. | I.P.+ | BaCP (old) | delta' (old) |
|---|---:|---:|---:|---:|
| resnet34 magnitude 0.95 | 90.65 | 91.44 | 91.47 | +0.03 |
| resnet34 magnitude 0.97 | 90.28 | 91.51 | 91.40 | -0.11 |

Records carry the `.sload` suffix, so they sit beside the reported numbers
rather than satisfying their keys.

**Learning rate stays at 0.1** -- that is the tuned value for this arm and is
not what this probe is testing.


In [ ]:
import sys, pathlib

here = pathlib.Path.cwd()
while not (here / '.git').exists() and here != here.parent:
    here = here.parent
sys.path.insert(0, str(here / 'project' / 'test_notebooks'))

import nb_common as nb
info = nb.setup()


## Plan

In [ ]:
MODEL      = 'resnet34'
PRUNER     = 'magnitude'
SPARSITIES = (0.95, 0.97)
SEED       = 1
GPU        = 0

plan = [nb.make_cell(MODEL, 'bacp', seed=SEED, pruner=PRUNER,
                     sparsity=sp, variant='sload')
        for sp in SPARSITIES]

# nothing but the student init may differ from the reported BaCP arm
ref = nb.FAMILIES[MODEL]['bacp']
for c in plan:
    for k in ('learning_rate', 'epochs', 'epochs_ft', 'delta_T',
              'sparsity_scheduler', 'recovery_epochs', 'val_split',
              'prune_task_head', 'wanda_group', 'optimizer_type',
              'batch_size', 'num_classes', 'dataset_name', 'tau'):
        if k in ref:
            assert c['config'][k] == ref[k], (c['key'], k, c['config'][k], ref[k])
    assert c['key'].endswith('.sload'), c['key']
    assert c['config']['learning_rate'] == 0.1, c['config']['learning_rate']

print('%d cells, ~%.0f min' % (len(plan), len(plan) * 8.03))
for c in plan:
    print('  ', c['key'])
assert nb.sanity_check(plan), 'sanity check failed'


## Run

In [ ]:
nb.run_group(plan, gpu=GPU)


## Verdict

In [ ]:
import json, glob, os

root = os.environ['BACP_RESULTS_DIR']
acc = {}
for f in glob.glob(os.path.join(root, 'runs', '*.json')):
    r = json.load(open(f, encoding='utf-8'))
    k = r.get('experiment_group') or ''
    if r.get('status') == 'ok':
        acc[k] = r.get('test_acc_exact_pct') or r.get('test_acc_pct')

OLD = {0.95: (90.65, 91.44, 91.47), 0.97: (90.28, 91.51, 91.40)}

print('%-9s %8s %8s %10s %10s %9s %9s' %
      ('sparsity', 'I.P.', 'I.P.+', 'BaCP old', 'BaCP new', "d' old", "d' new"))
print('-' * 70)
for sp in SPARSITIES:
    ip, ipp, old = OLD[sp]
    new = acc.get('static.bacp.%s.cifar10.s%s.%s.seed%d.sload'
                  % (MODEL, sp, PRUNER, SEED))
    if new is None:
        print('%-9s %8.2f %8.2f %10.2f %10s' % (sp, ip, ipp, old, 'pending'))
        continue
    print('%-9s %8.2f %8.2f %10.2f %10.2f %+9.2f %+9.2f'
          % (sp, ip, ipp, old, new, old - ipp, new - ipp))
print()
print("d' is against I.P.+, the budget-matched control.")
print('A material jump means the student init was suppressing the effect;')
print('no movement means the contrastive terms add nothing either way.')
